In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/datasets/shashwatkumarjha/bestweights/best.pt
/kaggle/input/datasets/shashwatkumarjha/newtrimvideo/ChotiClip.mp4
/kaggle/input/datasets/shashwatkumarjha/weightstrainedfinal/best.pt


Installing Dependencies


In [2]:
!pip install ultralytics opencv-python-headless

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 20.5 MB/s eta 0:00:00


In [3]:
import cv2
import numpy as np
from ultralytics import YOLO
from collections import defaultdict

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


In [4]:
# Load YOLOv8 model (you can later replace with your trained weights)
model = YOLO("/kaggle/input/datasets/shashwatkumarjha/weightstrainedfinal/best.pt", task='detect')

# Video path
video_path = "/kaggle/input/datasets/shashwatkumarjha/newtrimvideo/ChotiClip.mp4"

cap = cv2.VideoCapture(video_path)

# Video properties
fps = int(cap.get(cv2.CAP_PROP_FPS))
print("Actual FPS:", fps)
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

print("FPS:", fps)
print("Resolution:", width, "x", height)

Actual FPS: 100
FPS: 100
Resolution: 1920 x 1080


In [5]:
print(model.names)

{0: 'car', 1: 'person', 2: '2wheeler'}


In [6]:
import math

def is_valid(prev, curr, max_dist=60):
    if prev is None:
        return True
    dist = math.hypot(curr[0]-prev[0], curr[1]-prev[1])
    return dist < max_dist


def smooth(points, k=5):
    if len(points) < k:
        return points
    
    smoothed = []
    for i in range(len(points)):
        pts = points[max(0, i-k):i+1]
        x_avg = sum(p[0] for p in pts) / len(pts)
        y_avg = sum(p[1] for p in pts) / len(pts)
        smoothed.append((int(x_avg), int(y_avg)))
    
    return smoothed

In [7]:
roi_points = np.array([
    [540,1066],
    [972,216],
    [1080,216],
    [1650,1066]
], dtype=np.int32)

def apply_roi_mask(frame):
    mask = np.zeros_like(frame)
    cv2.fillPoly(mask, [roi_points], (255,255,255))
    masked = cv2.bitwise_and(frame, mask)
    return masked

In [8]:
import numpy as np
import cv2

# Your ROI points (same as masking)
src = np.array([
    [540,1066],
    [1650,1066],
    [1080,216],
    [972,216]
], dtype=np.float32)

# Real world coordinates (meters)
# width = 3.5m, length ≈ 20m
dst = np.array([
    [0, 20],
    [3.5, 20],
    [3.5, 0],
    [0, 0]
], dtype=np.float32)

# Compute homography
H, _ = cv2.findHomography(src, dst)

print("Homography matrix:\n", H)

Homography matrix:
 [[   -0.02387   -0.012132      25.822]
 [-4.9987e-17    -0.17812      38.475]
 [-2.5076e-18  -0.0080397           1]]


In [9]:
def pixel_to_world(x, y):
    pt = np.array([x, y, 1.0])  # 1D array (important)
    world = H @ pt              # matrix multiply

    world = world / world[2]    # normalize

    return float(world[0]), float(world[1])

In [10]:
world_history = defaultdict(list)
speed_dict = {}

Using Homograpphy


In [11]:
import cv2
import numpy as np
from collections import defaultdict
import math
from IPython.display import FileLink, display

# =========================
# CONFIG
# =========================
VEHICLE_CLASSES = [0, 2]
MAX_HISTORY = 30

# =========================
# COLORS
# =========================
COLOR_CAR = (0, 255, 255)
COLOR_2W = (255, 255, 0)
COLOR_TRAJ = (255, 0, 0)
COLOR_LINE = (0, 0, 255)

# =========================
# ROI
# =========================
roi_points = np.array([
    [540,1066],
    [972,216],
    [1080,216],
    [1650,1066]
], dtype=np.int32)

def apply_roi_mask(frame):
    mask = np.zeros_like(frame)
    cv2.fillPoly(mask, [roi_points], (255,255,255))
    return cv2.bitwise_and(frame, mask)

# =========================
# TRAJECTORY
# =========================
def is_valid(prev, curr, max_dist=60):
    if prev is None:
        return True
    return math.hypot(curr[0]-prev[0], curr[1]-prev[1]) < max_dist

def smooth(points, k=5):
    if len(points) < k:
        return points
    smoothed = []
    for i in range(len(points)):
        pts = points[max(0,i-k):i+1]
        smoothed.append((
            int(sum(p[0] for p in pts)/len(pts)),
            int(sum(p[1] for p in pts)/len(pts))
        ))
    return smoothed

# =========================
# WORLD CONVERSION
# =========================
def pixel_to_world(x, y):
    pt = np.array([x, y, 1.0])
    world = H @ pt
    world = world / world[2]
    return float(world[0]), float(world[1])

# =========================
# STORAGE
# =========================
track_history = defaultdict(list)
world_history = defaultdict(list)
speed_display = {}

# =========================
# SINGLE SPEED LINE
# =========================
speed_line_y = 700   # 🔥 adjust if needed

# =========================
# VIDEO IO
# =========================
cap = cv2.VideoCapture(video_path)

fps = int(cap.get(cv2.CAP_PROP_FPS))
if fps < 60:
    fps = 100

print("Using FPS:", fps)

output_path = "/kaggle/working/output_final.mp4"

out = cv2.VideoWriter(
    output_path,
    cv2.VideoWriter_fourcc(*"mp4v"),
    fps,
    (width, height)
)

# =========================
# MAIN LOOP
# =========================
while True:
    ret, frame = cap.read()
    if not ret:
        break

    masked = apply_roi_mask(frame)

    # Draw speed line
    cv2.line(frame, (0, speed_line_y), (width, speed_line_y), COLOR_LINE, 3)

    results = model.track(
        source=masked,
        conf=0.2,
        iou=0.5,
        tracker="bytetrack.yaml",
        persist=True
    )[0]

    if results.boxes is not None and results.boxes.id is not None:
        boxes = results.boxes.xyxy.cpu().numpy()
        ids = results.boxes.id.cpu().numpy().astype(int)
        classes = results.boxes.cls.cpu().numpy().astype(int)

        for box, tid, cls in zip(boxes, ids, classes):

            if cls not in VEHICLE_CLASSES:
                continue

            x1,y1,x2,y2 = map(int, box)
            cx = int((x1+x2)/2)
            cy = int(y2)

            color = COLOR_CAR if cls==0 else COLOR_2W

            # ================= WORLD TRACK =================
            wx, wy = pixel_to_world(cx, cy)
            world_history[tid].append((wx, wy))

            # ================= SPEED LOGIC =================
            if len(world_history[tid]) >= 6:

                # only compute when near line (best detection zone)
                if abs(cy - speed_line_y) < 50:

                    x_prev, y_prev = world_history[tid][-6]
                    x_curr, y_curr = world_history[tid][-1]

                    dist = ((x_curr-x_prev)**2 + (y_curr-y_prev)**2)**0.5

                    dt = 5 / fps
                    speed = (dist / dt) * 3.6

                    # smooth update
                    prev_spd = speed_display.get(tid, speed)

                    if abs(speed - prev_spd) > 3:
                        speed_display[tid] = speed
                    else:
                        speed_display[tid] = prev_spd

            # ================= TRAJECTORY =================
            prev = track_history[tid][-1] if track_history[tid] else None
            if is_valid(prev, (cx, cy)):
                track_history[tid].append((cx, cy))

            if len(track_history[tid]) > MAX_HISTORY:
                track_history[tid].pop(0)

            pts = smooth(track_history[tid])
            for i in range(1,len(pts)):
                cv2.line(frame, pts[i-1], pts[i], COLOR_TRAJ, 2)

            # ================= DRAW =================
            cv2.rectangle(frame,(x1,y1),(x2,y2),color,2)
            cv2.putText(frame,f"ID {tid}",(x1,y1-10),
                        cv2.FONT_HERSHEY_SIMPLEX,0.5,color,2)

            spd = speed_display.get(tid,0)
            cv2.putText(frame,f"{spd:.1f} km/h",(x1,y2+20),
                        cv2.FONT_HERSHEY_SIMPLEX,0.5,color,2)

    out.write(frame)

cap.release()
out.release()

print("✅ FINAL SPEED WORKING (ROBUST METHOD)")
display(FileLink(output_path))

Using FPS: 100
requirements: Ultralytics requirement ['lap>=0.5.12'] not found, attempting AutoUpdate...
Using Python 3.12.12 environment at: /usr
Resolved 2 packages in 256ms
 Downloaded lap
Prepared 1 package in 77ms
Installed 1 package in 1ms
 + lap==0.5.13

requirements: AutoUpdate success ✅ 1.0s
WARNING ⚠️ requirements: Restart runtime or rerun command for updates to take effect


0: 384x640 2 cars, 2 2wheelers, 133.1ms
Speed: 11.3ms preprocess, 133.1ms inference, 51.8ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 2 cars, 2 2wheelers, 24.9ms
Speed: 2.5ms preprocess, 24.9ms inference, 1.3ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 2 cars, 2 2wheelers, 25.0ms
Speed: 2.9ms preprocess, 25.0ms inference, 1.4ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 2 cars, 2 2wheelers, 25.0ms
Speed: 2.2ms preprocess, 25.0ms inference, 1.2ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 2 cars, 2 2wheelers, 25.0ms
Speed: 2.9ms preproc

/kaggle/working/output_final.mp4

Got the speed

In [12]:
import cv2
import numpy as np
from collections import defaultdict
import math
from IPython.display import FileLink, display

# =========================
# CONFIG
# =========================
VEHICLE_CLASSES = [0, 2]
MAX_HISTORY = 30

# =========================
# COLORS
# =========================
COLOR_CAR = (0, 255, 255)
COLOR_2W = (255, 255, 0)
COLOR_TRAJ = (255, 0, 0)
COLOR_ZONE = (0, 0, 255)

# =========================
# ROI
# =========================
roi_points = np.array([
    [540,1066],
    [972,216],
    [1080,216],
    [1650,1066]
], dtype=np.int32)

def apply_roi_mask(frame):
    mask = np.zeros_like(frame)
    cv2.fillPoly(mask, [roi_points], (255,255,255))
    return cv2.bitwise_and(frame, mask)

# =========================
# TRAJECTORY
# =========================
def is_valid(prev, curr, max_dist=60):
    if prev is None:
        return True
    return math.hypot(curr[0]-prev[0], curr[1]-prev[1]) < max_dist

def smooth(points, k=5):
    if len(points) < k:
        return points
    smoothed = []
    for i in range(len(points)):
        pts = points[max(0,i-k):i+1]
        smoothed.append((
            int(sum(p[0] for p in pts)/len(pts)),
            int(sum(p[1] for p in pts)/len(pts))
        ))
    return smoothed

# =========================
# STORAGE
# =========================
track_history = defaultdict(list)
pixel_history = defaultdict(list)
speed_display = {}

# =========================
# SPEED ZONE (NEAR CAMERA)
# =========================
zone_y_min = 600
zone_y_max = 900

PIXEL_TO_METER = 0.04   # 🔥 IMPORTANT (adjust later if needed)

# =========================
# VIDEO
# =========================
cap = cv2.VideoCapture(video_path)

fps = int(cap.get(cv2.CAP_PROP_FPS))
if fps < 60:
    fps = 100

print("Using FPS:", fps)

output_path = "/kaggle/working/output222_final.mp4"

out = cv2.VideoWriter(
    output_path,
    cv2.VideoWriter_fourcc(*"mp4v"),
    fps,
    (width, height)
)

# =========================
# MAIN LOOP
# =========================
while True:
    ret, frame = cap.read()
    if not ret:
        break

    masked = apply_roi_mask(frame)

    # draw zone
    cv2.rectangle(frame, (0, zone_y_min), (width, zone_y_max), COLOR_ZONE, 2)

    results = model.track(
        source=masked,
        conf=0.2,
        iou=0.5,
        tracker="bytetrack.yaml",
        persist=True
    )[0]

    if results.boxes is not None and results.boxes.id is not None:
        boxes = results.boxes.xyxy.cpu().numpy()
        ids = results.boxes.id.cpu().numpy().astype(int)
        classes = results.boxes.cls.cpu().numpy().astype(int)

        for box, tid, cls in zip(boxes, ids, classes):

            if cls not in VEHICLE_CLASSES:
                continue

            x1,y1,x2,y2 = map(int, box)
            cx = int((x1+x2)/2)
            cy = int(y2)

            color = COLOR_CAR if cls==0 else COLOR_2W

            # ================= SPEED =================
            if zone_y_min < cy < zone_y_max:

                pixel_history[tid].append((cx, cy))

                if len(pixel_history[tid]) >= 6:
                    x_prev, y_prev = pixel_history[tid][-6]
                    x_curr, y_curr = pixel_history[tid][-1]

                    pixel_dist = ((x_curr-x_prev)**2 + (y_curr-y_prev)**2)**0.5

                    dist = pixel_dist * PIXEL_TO_METER

                    dt = 5 / fps
                    speed = (dist / dt) * 3.6

                    prev_spd = speed_display.get(tid, speed)

                    if abs(speed - prev_spd) > 3:
                        speed_display[tid] = speed
                    else:
                        speed_display[tid] = prev_spd

            # ================= TRAJECTORY =================
            prev = track_history[tid][-1] if track_history[tid] else None

            if is_valid(prev, (cx, cy)):
                track_history[tid].append((cx, cy))

            if len(track_history[tid]) > MAX_HISTORY:
                track_history[tid].pop(0)

            pts = smooth(track_history[tid])
            for i in range(1,len(pts)):
                cv2.line(frame, pts[i-1], pts[i], COLOR_TRAJ, 2)

            # ================= DRAW =================
            cv2.rectangle(frame,(x1,y1),(x2,y2),color,2)
            cv2.putText(frame,f"ID {tid}",(x1,y1-10),
                        cv2.FONT_HERSHEY_SIMPLEX,0.5,color,2)

            spd = speed_display.get(tid,0)
            cv2.putText(frame,f"{spd:.1f} km/h",(x1,y2+20),
                        cv2.FONT_HERSHEY_SIMPLEX,0.5,color,2)

    out.write(frame)

cap.release()
out.release()

print("✅ FINAL SPEED (CALIBRATED ZONE METHOD)")
display(FileLink(output_path))

Using FPS: 100

0: 384x640 1 2wheeler, 24.9ms
Speed: 2.6ms preprocess, 24.9ms inference, 1.3ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 2 cars, 2 2wheelers, 25.0ms
Speed: 3.0ms preprocess, 25.0ms inference, 1.3ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 2 cars, 2 2wheelers, 25.0ms
Speed: 2.8ms preprocess, 25.0ms inference, 1.3ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 2 cars, 2 2wheelers, 24.9ms
Speed: 2.3ms preprocess, 24.9ms inference, 1.3ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 2 cars, 2 2wheelers, 24.9ms
Speed: 2.3ms preprocess, 24.9ms inference, 1.2ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 2 cars, 2 2wheelers, 24.9ms
Speed: 2.5ms preprocess, 24.9ms inference, 1.3ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 2 cars, 2 2wheelers, 24.9ms
Speed: 2.4ms preprocess, 24.9ms inference, 1.2ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 2 cars, 2 2wheelers, 24

/kaggle/working/output222_final.mp4

MTTC CSV and Video 

In [13]:
import cv2
import numpy as np
from collections import defaultdict
import math
import csv
from IPython.display import FileLink, display

# =========================
# CONFIG
# =========================
VEHICLE_CLASSES = [0, 2]
MAX_HISTORY = 30

COLOR_CAR = (0, 255, 255)
COLOR_2W = (255, 255, 0)
COLOR_TRAJ = (255, 0, 0)
COLOR_ZONE = (0, 0, 255)
COLOR_LINE = (0, 0, 255)

# =========================
# ROI
# =========================
roi_points = np.array([
    [540,1066],
    [972,216],
    [1080,216],
    [1650,1066]
], dtype=np.int32)

def apply_roi_mask(frame):
    mask = np.zeros_like(frame)
    cv2.fillPoly(mask, [roi_points], (255,255,255))
    return cv2.bitwise_and(frame, mask)

# =========================
# TRAJECTORY
# =========================
def is_valid(prev, curr, max_dist=60):
    if prev is None:
        return True
    return math.hypot(curr[0]-prev[0], curr[1]-prev[1]) < max_dist

def smooth(points, k=5):
    if len(points) < k:
        return points
    smoothed = []
    for i in range(len(points)):
        pts = points[max(0,i-k):i+1]
        smoothed.append((
            int(sum(p[0] for p in pts)/len(pts)),
            int(sum(p[1] for p in pts)/len(pts))
        ))
    return smoothed

# =========================
# STORAGE
# =========================
track_history = defaultdict(list)
pixel_history = defaultdict(list)

speed_display = {}
speed_buffer = defaultdict(list)
line_speed = {}
acceleration = {}

collision_pairs = set()
mttc_records = []

# =========================
# SPEED ZONE + LINE
# =========================
zone_y_min = 600
zone_y_max = 900
line_capture = 600   # 🔥 IMPORTANT

PIXEL_TO_METER = 0.04

# =========================
# VIDEO
# =========================
cap = cv2.VideoCapture(video_path)

fps = int(cap.get(cv2.CAP_PROP_FPS))
if fps < 60:
    fps = 100

print("Using FPS:", fps)

output_path = "/kaggle/working/output2222222222222mtmtc_final.mp4"

out = cv2.VideoWriter(
    output_path,
    cv2.VideoWriter_fourcc(*"mp4v"),
    fps,
    (width, height)
)

frame_count = 0

# =========================
# MAIN LOOP
# =========================
while True:
    ret, frame = cap.read()
    if not ret:
        break

    frame_count += 1
    time_now = frame_count / fps

    masked = apply_roi_mask(frame)

    # draw zone + line
    cv2.rectangle(frame, (0, zone_y_min), (width, zone_y_max), COLOR_ZONE, 2)
    cv2.line(frame, (0, line_capture), (width, line_capture), COLOR_LINE, 2)

    results = model.track(
        source=masked,
        conf=0.2,
        iou=0.5,
        tracker="bytetrack.yaml",
        persist=True
    )[0]

    current_ids = []

    if results.boxes is not None and results.boxes.id is not None:
        boxes = results.boxes.xyxy.cpu().numpy()
        ids = results.boxes.id.cpu().numpy().astype(int)
        classes = results.boxes.cls.cpu().numpy().astype(int)

        for box, tid, cls in zip(boxes, ids, classes):

            if cls not in VEHICLE_CLASSES:
                continue

            current_ids.append(tid)

            x1,y1,x2,y2 = map(int, box)
            cx = int((x1+x2)/2)
            cy = int(y2)

            color = COLOR_CAR if cls==0 else COLOR_2W

            # ================= SPEED (UNCHANGED CORE) =================
            if zone_y_min < cy < zone_y_max:

                pixel_history[tid].append((cx, cy))

                if len(pixel_history[tid]) >= 6:
                    x_prev, y_prev = pixel_history[tid][-6]
                    x_curr, y_curr = pixel_history[tid][-1]

                    pixel_dist = ((x_curr-x_prev)**2 + (y_curr-y_prev)**2)**0.5
                    dist = pixel_dist * PIXEL_TO_METER

                    dt = 5 / fps
                    speed = (dist / dt) * 3.6

                    prev_spd = speed_display.get(tid, speed)

                    if abs(speed - prev_spd) > 3:
                        speed_display[tid] = speed
                    else:
                        speed_display[tid] = prev_spd

                    # store for MTTC
                    speed_buffer[tid].append(speed_display[tid])

                    if len(speed_buffer[tid]) > 10:
                        speed_buffer[tid].pop(0)

                    # acceleration
                    if len(speed_buffer[tid]) >= 2:
                        v1 = speed_buffer[tid][-2]
                        v2 = speed_buffer[tid][-1]
                        acceleration[tid] = (v2 - v1) / (1/fps)

            # ================= CAPTURE SPEED AT LINE =================
            if abs(cy - line_capture) < 10:
                if tid not in line_speed and len(speed_buffer[tid]) > 0:
                    line_speed[tid] = max(speed_buffer[tid])

            # ================= TRAJECTORY =================
            prev = track_history[tid][-1] if track_history[tid] else None

            if is_valid(prev, (cx, cy)):
                track_history[tid].append((cx, cy))

            if len(track_history[tid]) > MAX_HISTORY:
                track_history[tid].pop(0)

            pts = smooth(track_history[tid])
            for i in range(1,len(pts)):
                cv2.line(frame, pts[i-1], pts[i], COLOR_TRAJ, 2)

            # ================= DRAW =================
            cv2.rectangle(frame,(x1,y1),(x2,y2),color,2)
            cv2.putText(frame,f"ID {tid}",(x1,y1-10),
                        cv2.FONT_HERSHEY_SIMPLEX,0.5,color,2)

            spd = speed_display.get(tid,0)
            cv2.putText(frame,f"{spd:.1f} km/h",(x1,y2+20),
                        cv2.FONT_HERSHEY_SIMPLEX,0.5,color,2)

    # ================= MTTC =================
    ids_list = list(line_speed.keys())

    for i in range(len(ids_list)):
        for j in range(i+1, len(ids_list)):

            id1 = ids_list[i]
            id2 = ids_list[j]

            if id1 not in acceleration or id2 not in acceleration:
                continue

            v1 = line_speed[id1] / 3.6
            v2 = line_speed[id2] / 3.6

            dv = v1 - v2
            da = acceleration[id1] - acceleration[id2]

            if abs(da) < 1e-3:
                continue

            if id1 in track_history and id2 in track_history:
                x1,y1 = track_history[id1][-1]
                x2,y2 = track_history[id2][-1]

                S = ((x1-x2)**2 + (y1-y2)**2)**0.5 * PIXEL_TO_METER
            else:
                continue

            disc = dv**2 + 2*da*S
            if disc < 0:
                continue

            t1 = (dv + math.sqrt(disc)) / da
            t2 = (dv - math.sqrt(disc)) / da

            candidates = [t for t in [t1,t2] if t > 0]

            if candidates:
                mttc = min(candidates)

                if mttc < 1.5:
                    collision_pairs.add((id1,id2))

                    mttc_records.append([
                        frame_count,
                        id1,
                        id2,
                        round(mttc,2)
                    ])

    # ================= ALERT =================
    for (a,b) in collision_pairs:
        if a in current_ids or b in current_ids:
            cv2.putText(frame,"⚠️ MTTC RISK",(50,50),
                        cv2.FONT_HERSHEY_SIMPLEX,1,(0,0,255),3)

    out.write(frame)

cap.release()
out.release()

# ================= SAVE CSV =================
csv_path = "/kaggle/working/mttc222222_results.csv"

with open(csv_path, "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(["Frame","Vehicle1","Vehicle2","MTTC(sec)"])
    writer.writerows(mttc_records)

print("✅ FINAL SYSTEM WITH CORRECT SPEED + MTTC")
display(FileLink(output_path))
display(FileLink(csv_path))

Using FPS: 100

0: 384x640 1 2wheeler, 25.0ms
Speed: 2.2ms preprocess, 25.0ms inference, 2.0ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 2 cars, 2 2wheelers, 25.0ms
Speed: 2.8ms preprocess, 25.0ms inference, 1.2ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 2 cars, 2 2wheelers, 24.9ms
Speed: 2.2ms preprocess, 24.9ms inference, 1.3ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 2 cars, 2 2wheelers, 24.9ms
Speed: 2.3ms preprocess, 24.9ms inference, 1.2ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 2 cars, 2 2wheelers, 22.0ms
Speed: 2.9ms preprocess, 22.0ms inference, 1.2ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 2 cars, 2 2wheelers, 22.0ms
Speed: 2.9ms preprocess, 22.0ms inference, 1.2ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 2 cars, 2 2wheelers, 22.0ms
Speed: 2.1ms preprocess, 22.0ms inference, 1.2ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 2 cars, 2 2wheelers, 22

/kaggle/working/output2222222222222mtmtc_final.mp4

/kaggle/working/mttc222222_results.csv

Heatmap and TTC

In [14]:
import cv2
import numpy as np
from collections import defaultdict
import math
import csv
from IPython.display import FileLink, display

# =========================
# CONFIG
# =========================
VEHICLE_CLASSES = [0, 2]
MAX_HISTORY = 30

COLOR_CAR = (0, 255, 255)
COLOR_2W = (255, 255, 0)
COLOR_TRAJ = (255, 0, 0)
COLOR_ZONE = (0, 0, 255)
COLOR_LINE = (0, 0, 255)

# =========================
# ROI
# =========================
roi_points = np.array([
    [540,1066],
    [972,216],
    [1080,216],
    [1650,1066]
], dtype=np.int32)

def apply_roi_mask(frame):
    mask = np.zeros_like(frame)
    cv2.fillPoly(mask, [roi_points], (255,255,255))
    return cv2.bitwise_and(frame, mask)

# =========================
# TRAJECTORY
# =========================
def is_valid(prev, curr, max_dist=60):
    if prev is None:
        return True
    return math.hypot(curr[0]-prev[0], curr[1]-prev[1]) < max_dist

def smooth(points, k=5):
    if len(points) < k:
        return points
    smoothed = []
    for i in range(len(points)):
        pts = points[max(0,i-k):i+1]
        smoothed.append((
            int(sum(p[0] for p in pts)/len(pts)),
            int(sum(p[1] for p in pts)/len(pts))
        ))
    return smoothed

# =========================
# STORAGE
# =========================
track_history = defaultdict(list)
pixel_history = defaultdict(list)

speed_display = {}
speed_buffer = defaultdict(list)
line_speed = {}
acceleration = {}

collision_pairs = set()
mttc_records = []

# 🔥 HEATMAP STORAGE
heatmap = np.zeros((height, width), dtype=np.float32)

# =========================
# SPEED ZONE + LINE
# =========================
zone_y_min = 600
zone_y_max = 900
line_capture = 600

PIXEL_TO_METER = 0.04

# =========================
# VIDEO
# =========================
cap = cv2.VideoCapture(video_path)

fps = int(cap.get(cv2.CAP_PROP_FPS))
if fps < 60:
    fps = 100

print("Using FPS:", fps)

output_path = "/kaggle/working/output1111111_final.mp4"

out = cv2.VideoWriter(
    output_path,
    cv2.VideoWriter_fourcc(*"mp4v"),
    fps,
    (width, height)
)

frame_count = 0

# =========================
# MAIN LOOP
# =========================
while True:
    ret, frame = cap.read()
    if not ret:
        break

    frame_count += 1
    time_now = frame_count / fps

    masked = apply_roi_mask(frame)

    # draw zone + line
    cv2.rectangle(frame, (0, zone_y_min), (width, zone_y_max), COLOR_ZONE, 2)
    cv2.line(frame, (0, line_capture), (width, line_capture), COLOR_LINE, 2)

    results = model.track(
        source=masked,
        conf=0.2,
        iou=0.5,
        tracker="bytetrack.yaml",
        persist=True
    )[0]

    current_ids = []

    if results.boxes is not None and results.boxes.id is not None:
        boxes = results.boxes.xyxy.cpu().numpy()
        ids = results.boxes.id.cpu().numpy().astype(int)
        classes = results.boxes.cls.cpu().numpy().astype(int)

        for box, tid, cls in zip(boxes, ids, classes):

            if cls not in VEHICLE_CLASSES:
                continue

            current_ids.append(tid)

            x1,y1,x2,y2 = map(int, box)
            cx = int((x1+x2)/2)
            cy = int(y2)

            color = COLOR_CAR if cls==0 else COLOR_2W

            # ================= SPEED =================
            if zone_y_min < cy < zone_y_max:

                pixel_history[tid].append((cx, cy))

                if len(pixel_history[tid]) >= 6:
                    x_prev, y_prev = pixel_history[tid][-6]
                    x_curr, y_curr = pixel_history[tid][-1]

                    pixel_dist = ((x_curr-x_prev)**2 + (y_curr-y_prev)**2)**0.5
                    dist = pixel_dist * PIXEL_TO_METER

                    dt = 5 / fps
                    speed = (dist / dt) * 3.6

                    prev_spd = speed_display.get(tid, speed)

                    if abs(speed - prev_spd) > 3:
                        speed_display[tid] = speed
                    else:
                        speed_display[tid] = prev_spd

                    speed_buffer[tid].append(speed_display[tid])

                    if len(speed_buffer[tid]) > 10:
                        speed_buffer[tid].pop(0)

                    if len(speed_buffer[tid]) >= 2:
                        v1 = speed_buffer[tid][-2]
                        v2 = speed_buffer[tid][-1]
                        acceleration[tid] = (v2 - v1) / (1/fps)

            # ================= CAPTURE SPEED =================
            if abs(cy - line_capture) < 10:
                if tid not in line_speed and len(speed_buffer[tid]) > 0:
                    line_speed[tid] = max(speed_buffer[tid])

            # ================= TRAJECTORY =================
            prev = track_history[tid][-1] if track_history[tid] else None

            if is_valid(prev, (cx, cy)):
                track_history[tid].append((cx, cy))

            if len(track_history[tid]) > MAX_HISTORY:
                track_history[tid].pop(0)

            pts = smooth(track_history[tid])
            for i in range(1,len(pts)):
                cv2.line(frame, pts[i-1], pts[i], COLOR_TRAJ, 2)

            # ================= DRAW =================
            cv2.rectangle(frame,(x1,y1),(x2,y2),color,2)
            cv2.putText(frame,f"ID {tid}",(x1,y1-10),
                        cv2.FONT_HERSHEY_SIMPLEX,0.5,color,2)

            spd = speed_display.get(tid,0)
            cv2.putText(frame,f"{spd:.1f} km/h",(x1,y2+20),
                        cv2.FONT_HERSHEY_SIMPLEX,0.5,color,2)

    # ================= MTTC =================
    ids_list = list(line_speed.keys())

    for i in range(len(ids_list)):
        for j in range(i+1, len(ids_list)):

            id1 = ids_list[i]
            id2 = ids_list[j]

            if id1 not in acceleration or id2 not in acceleration:
                continue

            v1 = line_speed[id1] / 3.6
            v2 = line_speed[id2] / 3.6

            dv = v1 - v2
            da = acceleration[id1] - acceleration[id2]

            if abs(da) < 1e-3:
                continue

            if id1 in track_history and id2 in track_history:
                x1,y1 = track_history[id1][-1]
                x2,y2 = track_history[id2][-1]

                S = ((x1-x2)**2 + (y1-y2)**2)**0.5 * PIXEL_TO_METER
            else:
                continue

            disc = dv**2 + 2*da*S
            if disc < 0:
                continue

            t1 = (dv + math.sqrt(disc)) / da
            t2 = (dv - math.sqrt(disc)) / da

            candidates = [t for t in [t1,t2] if t > 0]

            if candidates:
                mttc = min(candidates)

                if mttc < 1.5:
                    collision_pairs.add((id1,id2))

                    mttc_records.append([
                        frame_count,
                        id1,
                        id2,
                        round(mttc,2)
                    ])

                    # 🔥 HEATMAP UPDATE
                    xm = int((x1 + x2)/2)
                    ym = int((y1 + y2)/2)
                    cv2.circle(heatmap, (xm, ym), 20, 1, -1)

    # ================= ALERT =================
    for (a,b) in collision_pairs:
        if a in current_ids or b in current_ids:
            cv2.putText(frame,"⚠️ MTTC RISK",(50,50),
                        cv2.FONT_HERSHEY_SIMPLEX,1,(0,0,255),3)

    out.write(frame)

cap.release()
out.release()

# ================= SAVE CSV =================
csv_path = "/kaggle/working/mttc_1111111111results.csv"
with open(csv_path, "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(["Frame","Vehicle1","Vehicle2","MTTC(sec)"])
    writer.writerows(mttc_records)

# ================= SAVE HEATMAP =================
heatmap_norm = cv2.normalize(heatmap, None, 0, 255, cv2.NORM_MINMAX)
heatmap_uint8 = heatmap_norm.astype(np.uint8)
heatmap_color = cv2.applyColorMap(heatmap_uint8, cv2.COLORMAP_JET)

heatmap_path = "/kaggle/working/collision_heatmap.png"
cv2.imwrite(heatmap_path, heatmap_color)

print("✅ FINAL SYSTEM WITH HEATMAP + MTTC")

display(FileLink(output_path))
display(FileLink(csv_path))
display(FileLink(heatmap_path))

Using FPS: 100

0: 384x640 1 2wheeler, 25.0ms
Speed: 2.4ms preprocess, 25.0ms inference, 1.3ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 2 cars, 2 2wheelers, 24.9ms
Speed: 2.4ms preprocess, 24.9ms inference, 1.7ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 2 cars, 2 2wheelers, 25.0ms
Speed: 3.1ms preprocess, 25.0ms inference, 1.2ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 2 cars, 2 2wheelers, 24.9ms
Speed: 2.2ms preprocess, 24.9ms inference, 1.3ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 2 cars, 2 2wheelers, 24.3ms
Speed: 2.2ms preprocess, 24.3ms inference, 1.3ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 2 cars, 2 2wheelers, 22.0ms
Speed: 2.3ms preprocess, 22.0ms inference, 1.3ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 2 cars, 2 2wheelers, 22.0ms
Speed: 2.2ms preprocess, 22.0ms inference, 1.2ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 2 cars, 2 2wheelers, 22

/kaggle/working/output1111111_final.mp4

/kaggle/working/mttc_1111111111results.csv

/kaggle/working/collision_heatmap.png